# K-Means Clustering with PCA Dimensionality Reduction

This notebook combines **PCA** and **K-Means** clustering, first reducing the data dimensions, then clustering in the reduced space.

**Why use PCA before K-Means?**
1. **Noise reduction**: PCA removes less important dimensions that might confuse the clustering
2. **Speed**: Clustering on 16 dimensions is faster than 70+ dimensions
3. **Curse of dimensionality**: Distance calculations become less meaningful in very high dimensions

**Approach:**
1. Scale the data
2. Apply PCA to reduce to 16 components (~80% variance retained)
3. Run K-Means on the PCA-transformed data
4. Visualize results on PC1 vs PC2

## Import Libraries

In [ ]:
# Data manipulation and visualization
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning tools
from sklearn.preprocessing import StandardScaler  # For Z-score normalization
from sklearn.decomposition import PCA             # For dimensionality reduction
from sklearn.cluster import KMeans                # For clustering

## 1. Load and Prepare Data

Load the career data and set up the feature matrix. We remove any existing cluster columns to start fresh.

In [ ]:
# Load the prepared career data - change file path as needed
df = pd.read_csv('../Data/career_pivot_results.csv')

# Set job code and title as index
df.set_index(['O*NET-SOC Code', 'Title'], inplace=True)

# Remove any existing cluster columns from previous analyses
cols_to_drop = [c for c in df.columns if 'Cluster' in c]
df_clean = df.drop(columns=cols_to_drop)

# Create feature matrix X (all skills/abilities)
X = df_clean.values
print(f"Data shape: {X.shape}")  # (jobs, features)

## 2. Scale Data and Apply PCA

**Step 1: Scale** - Standardize features to mean=0, std=1

**Step 2: PCA** - Reduce from ~90 features to 16 principal components 
- 16 components capture ~80% of the variance (as seen in the 02_K-means_scaled File)

In [ ]:
# Step 1: Scale the data (Z-score normalization)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Step 2: Apply PCA to reduce to 16 dimensions
pca_16 = PCA(n_components=16)
X_pca_16 = pca_16.fit_transform(X_scaled)

print(f"PCA Done. Reduced dimensions from {X.shape[1]} to {X_pca_16.shape[1]}.")
print(f"Variance Explained: {np.sum(pca_16.explained_variance_ratio_):.2%}")

## 3. K-Means Clustering on PCA Data

In [ ]:
# Run K-Means on the PCA-reduced data
# K=6 clusters (based on elbow method)
kmeans_pca = KMeans(n_clusters=6, random_state=42, n_init=20)
labels = kmeans_pca.fit_predict(X_pca_16)  

# Save cluster labels back to the dataframe
df['PCA_KMeans_Cluster'] = labels

## 4. Evaluate Clusters

Check if the clusters make intuitive sense by looking at sample jobs from each cluster.

In [ ]:
# Display sample jobs from each cluster
print(f"*** CLUSTER RESULTS (K-Means on Top 16 PCs) ***\n")

for i in range(6):
    # Get all job titles in this cluster
    jobs = df[df['PCA_KMeans_Cluster'] == i].index.get_level_values('Title')
    
    # Show 8 random examples
    sample = np.random.choice(jobs, min(len(jobs), 8), replace=False)
    print(f"Cluster {i} ({len(jobs)} jobs): {', '.join(sample)}")

## 5. Visualize Clustering Results

Plot the clusters in 2D space on PC1 vs PC2.

In [ ]:
# Scatter plot: PC1 vs PC2, colored by cluster
plt.figure(figsize=(12, 8))
sns.scatterplot(
    x=X_pca_16[:, 0],   # PC1
    y=X_pca_16[:, 1],   # PC2
    hue=labels,          # Color by cluster
    palette='tab20',     # Color palette for clusters
    s=60,                # Point size
    alpha=0.7,           # Transparency
    legend='full'
)

plt.title('K-Means Clustering on 16 PCA Components (Visualized on PC1/PC2)')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='Cluster ID')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Evaluate Clustering Quality: Silhouette Score
The Silhouette Score measures how well-defined the clusters are:

- Ranges from -1 to +1
- +1: Points are well-matched to their own cluster and poorly-matched to others (good!)
- 0: Points are on cluster boundaries
- -1: Points may be assigned to the wrong cluster


In [ ]:
from sklearn.metrics import silhouette_score

score = silhouette_score(X_pca_16, kmeans_pca.labels_)

print(f"Silhouette Score: {score:.3f}")

---

## 📋 Summary: K-Means with PCA Dimensionality Reduction

### Best Parameters Found
| Parameter | Value | Rationale |
|-----------|-------|-----------|
| **PCA Components** | 16 | Captures ~80% of variance |
| **Number of Clusters (k)** | 6 | Elbow method |
| **Scaling** | StandardScaler | Required before PCA |


### Key Insights
1. **80% variance in 16 components** - significant dimensionality reduction
2. **Clustering in PCA space** can produce different (often better) results than raw feature space

### Limitations
- **Information loss**: ~20% of variance is discarded
- **Interpretability**: PCA components are linear combinations, harder to explain
- **2D visualization** shows only PC1/PC2 — clusters may overlap in 2D but be separated in 16D
- **Assumes linear relationships** — PCA may miss non-linear patterns

### Next Steps
- Check how the clusters perform on unscaled data since the range of values is on a standardized scale (file 04_K-means_unscaled)